> # **Capstone Project - QuizGenius AI**

# 1 - Project Introduction

## 1.1 - Project Introduction


In this hands-on project, we will implement a **Quiz Generation App** with the **Gemini Large Language Model**. This will inspire you to implement your own project ideas with LLMs.

### Key Features:





*   **Dynamic Quiz Generation**: Creates a 3-question multiple-choice quiz on any user-specified topic.
*   **Intermediate Difficulty**: Ensures questions are challenging yet accessible.
*   **Structured JSON Output**: The AI model returns quiz data in a consistent, easy-to-parse JSON format.
*   **Interactive User Interface**: Allows users to answer questions and provides immediate feedback, including explanations for correct answers.
*   **API Key Management**: Securely handles API keys using Colab's user data feature.

This project serves as a practical example of how generative AI can be integrated into interactive applications to create engaging and educational experiences.

# 2 - Project Setup

## 2.1 - Package Installation & API Setup

In [ ]:
# Install the Google GenAI library only when it is missing.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("google.genai") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-genai"])

print("✅ Libraries installed successfully!")

In [ ]:
import json
from google import genai
from google.genai import types
import re
import os

# 🔑 SETUP: read the key from the environment or Colab Secrets.
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        GEMINI_API_KEY = None

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

if client:
    print(f"✅ AI client is ready with model: {MODEL_NAME}")
else:
    print("⚠️ Set GEMINI_API_KEY in the environment or Colab Secrets to run the live quiz.")

In [ ]:
if client:
    print('Available models that support content generation:')
    for model in client.models.list():
        if 'generateContent' in (model.supported_actions or []):
            print(model.name)
else:
    print('Model listing skipped because GEMINI_API_KEY is not configured.')

# 3 - Prompt Engineering

## 3.1 - Creating the Prompt Template

In [ ]:
PROMPT_TEMPLATE = """
You are a Quiz Master. Generate a 3-question multiple-choice quiz about: "{topic}".

RULES:
1. Difficulty: Intermediate.
2. Provide 4 options (A, B, C, D) for each question.
3. Mark the correct answer clearly.
4. Provide a short explanation for the correct answer.

OUTPUT FORMAT:
Return raw JSON with this structure:
{{
  "quiz_title": "A catchy title",
  "questions": [
    {{
      "id": 1,
      "text": "The question text?",
      "options": {{
        "A": "Option A",
        "B": "Option B",
        "C": "Option C",
        "D": "Option D"
      }},
      "correct_option": "B",
      "explanation": "Why B is correct..."
    }}
  ]
}}
"""

print("✅ Prompt template defined.")

# 4 - Logic Development

## 4.1 - Building the Quiz Generation Logic

In [ ]:
def generate_quiz(topic):
    print(f"🧠 Generating quiz on '{topic}'...")

    try:
        if client is None:
            raise RuntimeError("GEMINI_API_KEY is not configured.")

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=PROMPT_TEMPLATE.format(topic=topic),
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )
        raw_json_text = response.text

        # First, try to extract JSON from markdown block if present
        # This regex handles both ```json {content} ``` and ``` {content} ```
        match = re.search(r"```(?:json)?\s*({.*})\s*```", raw_json_text, re.DOTALL)
        if match:
            json_string = match.group(1)
        else:
            json_string = raw_json_text

        # Remove invalid control characters (non-printable ASCII except valid JSON whitespace: \t, \n, \r)
        # Regex to match non-whitespace control characters (0x00-0x08, 0x0B, 0x0C, 0x0E-\x1F)
        cleaned_json_string = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_string)

        return json.loads(cleaned_json_string)

    except json.JSONDecodeError as e:
        print(f"❌ JSON Decode Error: {e}")
        print(f"   Problematic response text (truncated): {raw_json_text[:500]}...") # Print a portion of the problematic text
        return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ Quiz Generator function ready.")

## 4.2 - Main Application Runner

In [ ]:
import time

def run_quiz_app(topic=None, answers=None, pause_seconds=1):
    # 1. Get Topic
    if topic is None:
        topic = input("🎓 What topic do you want to be quizzed on? ")
    else:
        print(f"🎓 Quiz topic: {topic}")
    if not topic:
        return
    answer_iter = iter(answers) if answers is not None else None

    # 2. Generate Quiz
    quiz_data = generate_quiz(topic)
    if not quiz_data:
        print("Failed to generate quiz.")
        return

    # 3. Start Game
    print(f"\n✨ {quiz_data['quiz_title']} ✨")
    print("="*40)

    score = 0
    total = len(quiz_data['questions'])

    for q in quiz_data['questions']:
        print(f"\n❓ Q{q['id']}: {q['text']}")

        # Print Options
        for key, value in q['options'].items():
            print(f"   [{key}] {value}")

        # Get Answer
        if answer_iter is None:
            user_ans = input("   Your Answer (A/B/C/D): ").strip().upper()
        else:
            user_ans = next(answer_iter, "").strip().upper()
            print(f"   Your Answer (A/B/C/D): {user_ans}")

        # Check Answer
        if user_ans == q['correct_option']:
            print("   ✅ CORRECT!")
            score += 1
        else:
            print(f"   ❌ WRONG. The answer was {q['correct_option']}.")

        print(f"   💡 {q['explanation']}")
        time.sleep(pause_seconds) # Pause for effect

    # 4. Final Score
    print("\n" + "="*40)
    print(f"🏆 FINAL SCORE: {score}/{total}")
    if score == total:  # all answers were correct
        print("   Perfect Score! You are a master.")
    elif score > 0:
        print("   Good job! Keep studying.")
    else:
        print("   Better luck next time!")

    return score

# 5 - Final Demo

## 5.1 - Demo the application

In [ ]:
# Run the app
if client:
    if os.getenv("QUIZGENIUS_AUTORUN") == "1":
        demo_topic = os.getenv("QUIZGENIUS_TOPIC", "Natural Language Processing")
        run_quiz_app(topic=demo_topic, answers=["A", "A", "A"], pause_seconds=0)
    else:
        run_quiz_app()
else:
    print("Demo skipped: configure GEMINI_API_KEY, then run this cell again.")

# THANK YOU